# Telco Churn — Frozen Models - Optuna Tuning 
This notebook tunes and evaluates four predictive models:

- XGBoost (`xgboost.XGBClassifier`) — one-hot encoded categoricals
- Random Forest (`sklearn.ensemble.RandomForestClassifier`) — one-hot encoded categoricals
- GBM (`sklearn.ensemble.GradientBoostingClassifier`) — one-hot encoded categoricals
- CatBoost (`catboost.CatBoostClassifier`) — **native categorical features**

CatBoost is intentionally kept on native categorical variables rather than forced through OHE. This provides an additional robustness check on both model family and categorical representation.

## Core methodological principle

The primary intervention impact analysis is centered on **actual churners correctly identified at baseline (True Positives)**:

$$
TP = \{i : Y_i = 1 \land \hat{Y}^{baseline}_i = 1\}
$$

False Positives are analyzed separately and are not counted as rescued customers.

## Experimental design

- 70% train: Optuna optimization with 5-fold stratified CV
- 15% validation: one frozen threshold per tuned model
- 15% test: baseline evaluation and intervention simulation
- same train/validation/test split for all four models
- same intervention scenarios for all four models

## Important preprocessing difference

XGBoost, Random Forest, and GBM use:

$$
categorical \rightarrow imputation \rightarrow one\text{-}hot\ encoding
$$

CatBoost uses:

$$
original\ categorical\ variables \rightarrow native\ CatBoost\ handling
$$

The intervention logic always modifies the original customer-level variables before prediction.

## 0. Installation

In [1]:
# %pip install -U pandas openpyxl scikit-learn xgboost catboost optuna joblib matplotlib

In [2]:
import os
import re, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

SEED = 20260906
TARGET = "Churn"
ID_COL = "CustomerID"
NFOLDS = 5

N_TRIALS_XGB = 60
N_TRIALS_RF = 60
N_TRIALS_GBM = 60
N_TRIALS_CAT = 60

THRESHOLD_METHOD = "f1"   # or "youden"
MONTHLY_CHARGES_REDUCTION_PCT = 0.10
APPLY_CANDIDATE_ACTIONS = False

# Avoid nested over-parallelization inside CV.
N_JOBS_MODEL = -1
from joblib import Parallel, delayed

# CatBoost CV parallelism:
# Keep Optuna trials sequential for reproducible TPE search,
# but evaluate CV folds concurrently.
CPU_COUNT = os.cpu_count() or 1
CATBOOST_PARALLEL_FOLDS = min(NFOLDS, max(1, CPU_COUNT // 2))
CATBOOST_THREADS_PER_FOLD = max(
    1,
    CPU_COUNT // CATBOOST_PARALLEL_FOLDS
)

print("CPU_COUNT:", CPU_COUNT)
print("CatBoost parallel CV folds:", CATBOOST_PARALLEL_FOLDS)
print("CatBoost threads per fold:", CATBOOST_THREADS_PER_FOLD)

CPU_COUNT: 20
CatBoost parallel CV folds: 5
CatBoost threads per fold: 4


## 1. Load original Telco data and Recommended Actions

In [3]:
TELCO_FILE = "Telco-Customer-Churn.xlsx"
ACTIONS_FILE = "Recommended Actions.xlsx"

telco = pd.read_excel(TELCO_FILE, sheet_name="Dataset")
actions = pd.read_excel(ACTIONS_FILE)

telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")

required = {"Source","Sign","Action_Target","Recommended_Action","Global Rank","Global Score"}
missing = required - set(actions.columns)
if missing:
    raise ValueError(f"Missing Recommended Actions columns: {sorted(missing)}")

bad_targets = sorted(set(actions["Action_Target"].dropna().astype(str)) - set(telco.columns))
if bad_targets:
    raise ValueError("Action_Target not found in Telco-Customer: " + ", ".join(bad_targets))

print("Telco:", telco.shape)
print("Actions:", actions.shape)
display(actions)

Telco: (7043, 21)
Actions: (15, 8)


,Source,Sign,Actionable,Action_Target,Recommended_Action,Rationale,Global Rank,Global Score
0,InternetService_Fiber_optic,+,YES,InternetService,InternetService = DSL (candidate),Fiber optic has a positive churn edge. No inte...,1,0.555556
1,Contract_Month-to-month,+,YES,Contract,Contract = One year,"Month-to-month increases churn, while One-year...",2,0.500000
2,OnlineBackup_Yes,-,YES,OnlineBackup,OnlineBackup = Yes,"Yes has a negative edge to Churn, so recommend...",3,0.333333
3,MonthlyCharges,+,YES,MonthlyCharges,Reduce MonthlyCharges,Higher MonthlyCharges has a positive edge to C...,3,0.333333
4,PhoneService,-,YES,PhoneService,PhoneService = Yes,PhoneService has a negative edge to Churn; rec...,5,0.250000
5,MultipleLines_No_phone_service,+,NO,PhoneService,PhoneService = Yes,No phone service is a churn-increasing state; ...,5,0.250000
6,InternetService_No,+,YES,InternetService,InternetService = DSL (candidate),Move away from No internet service. Fiber opti...,7,0.166667
7,PaymentMethod_Mailed_check,-,YES,PaymentMethod,PaymentMethod = Mailed check,Mailed check has the strongest negative Paymen...,7,0.166667
8,PaymentMethod_Credit_card,-,YES,PaymentMethod,PaymentMethod = Mailed check,"Credit card is protective, but Mailed check ha...",9,0.083333
9,TechSupport_No_internet_service,+,NO,InternetService,InternetService = DSL (candidate),This state results from having no InternetServ...,9,0.083333


## 2. HIGH / MEDIUM / LOW from Global Score

- HIGH: score >= P75
- MEDIUM: P50 <= score < P75
- LOW: score < P50

Ties are preserved automatically.

In [4]:
p50 = actions["Global Score"].quantile(0.50)
p75 = actions["Global Score"].quantile(0.75)

def classify_priority(score):
    if score >= p75:
        return "HIGH"
    if score >= p50:
        return "MEDIUM"
    return "LOW"

actions["Priority"] = actions["Global Score"].apply(classify_priority)

print(f"P50={p50:.6f}, P75={p75:.6f}")
display(actions.sort_values(["Global Score","Global Rank"], ascending=[False,True])[
    ["Source","Sign","Action_Target","Recommended_Action","Global Score","Global Rank","Priority"]
])
actions.to_excel("Recommended Actions Priority.xlsx", index=False)

P50=0.166667, P75=0.291667


,Source,Sign,Action_Target,Recommended_Action,Global Score,Global Rank,Priority
0,InternetService_Fiber_optic,+,InternetService,InternetService = DSL (candidate),0.555556,1,HIGH
1,Contract_Month-to-month,+,Contract,Contract = One year,0.500000,2,HIGH
2,OnlineBackup_Yes,-,OnlineBackup,OnlineBackup = Yes,0.333333,3,HIGH
3,MonthlyCharges,+,MonthlyCharges,Reduce MonthlyCharges,0.333333,3,HIGH
4,PhoneService,-,PhoneService,PhoneService = Yes,0.250000,5,MEDIUM
5,MultipleLines_No_phone_service,+,PhoneService,PhoneService = Yes,0.250000,5,MEDIUM
6,InternetService_No,+,InternetService,InternetService = DSL (candidate),0.166667,7,MEDIUM
7,PaymentMethod_Mailed_check,-,PaymentMethod,PaymentMethod = Mailed check,0.166667,7,MEDIUM
8,PaymentMethod_Credit_card,-,PaymentMethod,PaymentMethod = Mailed check,0.083333,9,LOW
9,TechSupport_No_internet_service,+,InternetService,InternetService = DSL (candidate),0.083333,9,LOW


## 3. Fixed stratified 70/15/15 split

In [5]:
train_df, temp_df = train_test_split(
    telco, test_size=0.30, stratify=telco[TARGET], random_state=SEED
)
valid_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[TARGET], random_state=SEED
)

for name, df in [("train",train_df),("validation",valid_df),("test",test_df)]:
    print(name, df.shape, df[TARGET].value_counts(normalize=True).round(4).to_dict())

train (4930, 21) {'No': 0.7347, 'Yes': 0.2653}
validation (1056, 21) {'No': 0.7348, 'Yes': 0.2652}
test (1057, 21) {'No': 0.7342, 'Yes': 0.2658}


## 4. Preprocessing pipeline

The intervention is performed on the **original categorical test data**.

For modeling, preprocessing is embedded inside each scikit-learn pipeline:
- numeric median imputation;
- categorical most-frequent imputation;
- one-hot encoding.

Because preprocessing is inside the pipeline, it is re-fit independently inside each CV fold, preventing preprocessing leakage.

In [6]:
X_train = train_df.drop(columns=[TARGET, ID_COL]).copy()
X_valid = valid_df.drop(columns=[TARGET, ID_COL]).copy()
X_test  = test_df.drop(columns=[TARGET, ID_COL]).copy()

y_train = train_df[TARGET].astype(str).str.lower().eq("yes").astype(int)
y_valid = valid_df[TARGET].astype(str).str.lower().eq("yes").astype(int)
y_test  = test_df[TARGET].astype(str).str.lower().eq("yes").astype(int)

categorical_cols = X_train.select_dtypes(include=["object","category","bool"]).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

def make_preprocessor():
    num = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    cat = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    return ColumnTransformer([
        ("num", num, numeric_cols),
        ("cat", cat, categorical_cols)
    ])

cv = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

Numeric: ['SeniorCitizen', 'Tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: ['Gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 5. Explicit Optuna search spaces

In [7]:
SEARCH_SPACES = {
    "XGBoost": {
        "n_estimators": "200..1200 step 50",
        "max_depth": "2..10",
        "learning_rate": "0.005..0.20 log",
        "min_child_weight": "1..15",
        "subsample": "0.60..1.00",
        "colsample_bytree": "0.60..1.00",
        "gamma": "0..5",
        "reg_alpha": "1e-8..10 log",
        "reg_lambda": "1e-4..20 log"
    },
    "RandomForest": {
        "n_estimators": "200..1500 step 50",
        "max_depth": "4..40",
        "min_samples_split": "2..30",
        "min_samples_leaf": "1..15",
        "max_features": "sqrt, log2, 0.4, 0.6, 0.8",
        "bootstrap": "True, False",
        "class_weight": "None, balanced, balanced_subsample"
    },
    "GBM": {
        "n_estimators": "100..1200 step 50",
        "learning_rate": "0.005..0.20 log",
        "max_depth": "1..6",
        "min_samples_split": "2..30",
        "min_samples_leaf": "1..20",
        "subsample": "0.60..1.00",
        "max_features": "None, sqrt, log2"
    }
}

search_spaces_df = pd.DataFrame([
    {"Model":m, "Hyperparameter":p, "Search_Space":v}
    for m, space in SEARCH_SPACES.items()
    for p, v in space.items()
])
display(search_spaces_df)

,Model,Hyperparameter,Search_Space
0,XGBoost,n_estimators,200..1200 step 50
1,XGBoost,max_depth,2..10
2,XGBoost,learning_rate,0.005..0.20 log
3,XGBoost,min_child_weight,1..15
4,XGBoost,subsample,0.60..1.00
5,XGBoost,colsample_bytree,0.60..1.00
6,XGBoost,gamma,0..5
7,XGBoost,reg_alpha,1e-8..10 log
8,XGBoost,reg_lambda,1e-4..20 log
9,RandomForest,n_estimators,200..1500 step 50


## 6. Optuna objective functions — 5-fold CV ROC AUC

In [8]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200, step=50),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.20, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 15.0),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.60, 1.00),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 20.0, log=True)
    }
    estimator = XGBClassifier(
        **params, objective="binary:logistic", eval_metric="logloss",
        tree_method="hist", random_state=SEED, n_jobs=N_JOBS_MODEL
    )
    pipe = Pipeline([("preprocessor", make_preprocessor()), ("model", estimator)])
    scores = cross_val_score(pipe, X_train, y_train, scoring="roc_auc", cv=cv, n_jobs=1)
    trial.set_user_attr("cv_auc_std", float(scores.std()))
    return float(scores.mean())

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1500, step=50),
        "max_depth": trial.suggest_int("max_depth", 4, 40),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
        "max_features": trial.suggest_categorical("max_features", ["sqrt","log2",0.4,0.6,0.8]),
        "bootstrap": trial.suggest_categorical("bootstrap", [True,False]),
        "class_weight": trial.suggest_categorical("class_weight", [None,"balanced","balanced_subsample"])
    }
    estimator = RandomForestClassifier(**params, random_state=SEED, n_jobs=N_JOBS_MODEL)
    pipe = Pipeline([("preprocessor", make_preprocessor()), ("model", estimator)])
    scores = cross_val_score(pipe, X_train, y_train, scoring="roc_auc", cv=cv, n_jobs=1)
    trial.set_user_attr("cv_auc_std", float(scores.std()))
    return float(scores.mean())

def objective_gbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1200, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.20, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 6),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "subsample": trial.suggest_float("subsample", 0.60, 1.00),
        "max_features": trial.suggest_categorical("max_features", [None,"sqrt","log2"])
    }
    estimator = GradientBoostingClassifier(**params, random_state=SEED)
    pipe = Pipeline([("preprocessor", make_preprocessor()), ("model", estimator)])
    scores = cross_val_score(pipe, X_train, y_train, scoring="roc_auc", cv=cv, n_jobs=1)
    trial.set_user_attr("cv_auc_std", float(scores.std()))
    return float(scores.mean())

## 7. Run Optuna tuning

In [9]:
study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED), study_name="telco_xgboost")
study_rf  = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED+1), study_name="telco_random_forest")
study_gbm = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED+2), study_name="telco_gbm")

study_xgb.optimize(objective_xgb, n_trials=N_TRIALS_XGB, show_progress_bar=True)
study_rf.optimize(objective_rf, n_trials=N_TRIALS_RF, show_progress_bar=True)
study_gbm.optimize(objective_gbm, n_trials=N_TRIALS_GBM, show_progress_bar=True)

studies = {"XGBoost":study_xgb, "RandomForest":study_rf, "GBM":study_gbm}

[I 2026-09-08 12:42:57,011] A new study created in memory with name: telco_xgboost
[I 2026-09-08 12:42:57,011] A new study created in memory with name: telco_random_forest
[I 2026-09-08 12:42:57,014] A new study created in memory with name: telco_gbm


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-09-08 12:43:00,756] Trial 0 finished with value: 0.8433293114923824 and parameters: {'n_estimators': 1000, 'max_depth': 10, 'learning_rate': 0.050377768347647255, 'min_child_weight': 5.033570299922129, 'subsample': 0.8121309594346829, 'colsample_bytree': 0.7727358020941739, 'gamma': 2.383366919330751, 'reg_alpha': 0.023539397104594198, 'reg_lambda': 0.014638490942602033}. Best is trial 0 with value: 0.8433293114923824.
[I 2026-09-08 12:43:04,852] Trial 1 finished with value: 0.8463513654639641 and parameters: {'n_estimators': 450, 'max_depth': 6, 'learning_rate': 0.006298061942252456, 'min_child_weight': 5.488753682850573, 'subsample': 0.8263867656141387, 'colsample_bytree': 0.7793053946426974, 'gamma': 1.4259557219482715, 'reg_alpha': 2.706690438732483e-07, 'reg_lambda': 0.0008199345457077645}. Best is trial 1 with value: 0.8463513654639641.
[I 2026-09-08 12:43:05,725] Trial 2 finished with value: 0.8467801162152309 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learni

  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-09-08 12:46:01,404] Trial 0 finished with value: 0.8463663811581614 and parameters: {'n_estimators': 850, 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 13, 'max_features': 'sqrt', 'bootstrap': True, 'class_weight': None}. Best is trial 0 with value: 0.8463663811581614.
[I 2026-09-08 12:46:13,856] Trial 1 finished with value: 0.841535981516963 and parameters: {'n_estimators': 1450, 'max_depth': 7, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.4, 'bootstrap': False, 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.8463663811581614.
[I 2026-09-08 12:46:18,806] Trial 2 finished with value: 0.8425021479834804 and parameters: {'n_estimators': 450, 'max_depth': 4, 'min_samples_split': 29, 'min_samples_leaf': 5, 'max_features': 0.4, 'bootstrap': True, 'class_weight': None}. Best is trial 0 with value: 0.8463663811581614.
[I 2026-09-08 12:46:24,093] Trial 3 finished with value: 0.8442305588412286 and parameters: {'n_estimators': 450,

  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-09-08 12:55:10,415] Trial 0 finished with value: 0.839706675189831 and parameters: {'n_estimators': 750, 'learning_rate': 0.010998286740825387, 'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 8, 'subsample': 0.6963500859858628, 'max_features': None}. Best is trial 0 with value: 0.839706675189831.
[I 2026-09-08 12:55:17,643] Trial 1 finished with value: 0.8490617342227618 and parameters: {'n_estimators': 1000, 'learning_rate': 0.09293399428373392, 'max_depth': 1, 'min_samples_split': 24, 'min_samples_leaf': 14, 'subsample': 0.8044833373230854, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.8490617342227618.
[I 2026-09-08 12:55:19,472] Trial 2 finished with value: 0.8453798845122504 and parameters: {'n_estimators': 150, 'learning_rate': 0.017091140678314787, 'max_depth': 3, 'min_samples_split': 14, 'min_samples_leaf': 16, 'subsample': 0.7031671427045287, 'max_features': 'log2'}. Best is trial 1 with value: 0.8490617342227618.
[I 2026-09-08 12:55:40,890] Trial 

## CatBoost — native categorical Optuna tuning

CatBoost is trained on the original predictor columns, with categorical columns kept as strings.

To preserve the same 5-fold validation design, every Optuna trial still evaluates exactly the same five stratified folds and the same hyperparameter configuration. The only change is computational:

- the **five CV folds are evaluated concurrently** using `joblib`;
- Optuna trials themselves remain **sequential** (`n_jobs=1`);
- CPU threads are divided among simultaneously running folds to avoid oversubscription.

This preserves the Optuna TPE search sequence and does not reduce:
- the number of trials;
- the number of CV folds;
- the search space;
- the CatBoost iteration range.

Therefore, the change is intended as a computational acceleration rather than a reduction in optimization effort.

For every fold:
- numeric missing values are median-imputed using the training fold only;
- categorical missing values are replaced with a fixed string;
- CatBoost receives the categorical column indices directly;
- ROC AUC is computed on the held-out fold.

In [10]:
catboost_num_cols = [
    c for c in X_train.columns
    if pd.api.types.is_numeric_dtype(X_train[c])
]

catboost_cat_cols = [
    c for c in X_train.columns
    if c not in catboost_num_cols
]


def prepare_catboost_fold(X_fit, X_eval):
    X_fit_cb = X_fit.copy()
    X_eval_cb = X_eval.copy()

    # Fold-specific numeric imputation
    medians = X_fit_cb[catboost_num_cols].median()

    X_fit_cb[catboost_num_cols] = (
        X_fit_cb[catboost_num_cols]
        .fillna(medians)
        .astype(float)
    )

    X_eval_cb[catboost_num_cols] = (
        X_eval_cb[catboost_num_cols]
        .fillna(medians)
        .astype(float)
    )

    # CatBoost categorical features should not contain NaN
    for c in catboost_cat_cols:
        X_fit_cb[c] = (
            X_fit_cb[c]
            .fillna("__MISSING__")
            .astype(str)
        )
        X_eval_cb[c] = (
            X_eval_cb[c]
            .fillna("__MISSING__")
            .astype(str)
        )

    return X_fit_cb, X_eval_cb


def evaluate_catboost_fold(
    fit_idx,
    eval_idx,
    params
):
    X_fit = X_train.iloc[fit_idx].copy()
    X_eval = X_train.iloc[eval_idx].copy()
    y_fit = y_train.iloc[fit_idx]
    y_eval = y_train.iloc[eval_idx]

    X_fit_cb, X_eval_cb = prepare_catboost_fold(
        X_fit,
        X_eval
    )

    cat_idx = [
        X_fit_cb.columns.get_loc(c)
        for c in catboost_cat_cols
    ]

    fold_params = params.copy()
    fold_params["thread_count"] = CATBOOST_THREADS_PER_FOLD

    model = CatBoostClassifier(
        **fold_params
    )

    model.fit(
        X_fit_cb,
        y_fit,
        cat_features=cat_idx
    )

    probs = model.predict_proba(
        X_eval_cb
    )[:, 1]

    return float(
        roc_auc_score(
            y_eval,
            probs
        )
    )


def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int(
            "iterations", 200, 1200, step=50
        ),
        "depth": trial.suggest_int(
            "depth", 3, 10
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.20, log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1e-3, 30.0, log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.0, 5.0
        ),
        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0.0, 5.0
        ),
        "border_count": trial.suggest_int(
            "border_count", 32, 255
        ),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "verbose": 0,
        "random_seed": SEED,
        "allow_writing_files": False
    }

    # Materialize the exact same CV splits as before.
    splits = list(
        cv.split(
            X_train,
            y_train
        )
    )

    # Parallelize folds, not Optuna trials.
    scores = Parallel(
        n_jobs=CATBOOST_PARALLEL_FOLDS,
        backend="loky"
    )(
        delayed(evaluate_catboost_fold)(
            fit_idx,
            eval_idx,
            params
        )
        for fit_idx, eval_idx in splits
    )

    scores = np.asarray(
        scores,
        dtype=float
    )

    trial.set_user_attr(
        "cv_auc_std",
        float(scores.std())
    )

    return float(
        scores.mean()
    )


study_cat = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=SEED + 3
    ),
    study_name="CatBoost"
)

study_cat.optimize(
    objective_catboost,
    n_trials=N_TRIALS_CAT,
    n_jobs=1
)

print("Best CatBoost CV AUC:", study_cat.best_value)
print("Best CatBoost params:", study_cat.best_params)

[I 2026-09-08 13:20:39,635] A new study created in memory with name: CatBoost
[I 2026-09-08 13:21:45,287] Trial 0 finished with value: 0.842633057573611 and parameters: {'iterations': 1150, 'depth': 7, 'learning_rate': 0.006990957096953664, 'l2_leaf_reg': 0.005869436510232443, 'random_strength': 0.4741827374202434, 'bagging_temperature': 3.8368513737973604, 'border_count': 210}. Best is trial 0 with value: 0.842633057573611.
[I 2026-09-08 13:23:02,500] Trial 1 finished with value: 0.8357897160248553 and parameters: {'iterations': 1200, 'depth': 8, 'learning_rate': 0.027907516690766444, 'l2_leaf_reg': 7.0581585116281405, 'random_strength': 4.449918668562235, 'bagging_temperature': 1.7387488185567364, 'border_count': 158}. Best is trial 0 with value: 0.842633057573611.
[I 2026-09-08 13:23:50,041] Trial 2 finished with value: 0.8185502463303337 and parameters: {'iterations': 1100, 'depth': 6, 'learning_rate': 0.08520050594338133, 'l2_leaf_reg': 0.40879735024963076, 'random_strength': 2.16

Best CatBoost CV AUC: 0.8482600576990201
Best CatBoost params: {'iterations': 1050, 'depth': 3, 'learning_rate': 0.022952345403366905, 'l2_leaf_reg': 1.8393435360845845, 'random_strength': 3.4858913962894373, 'bagging_temperature': 4.2688675051144465, 'border_count': 112}


## 8. Best hyperparameters selected by Optuna

In [11]:
best_rows = []
best_long = []

for model_name, study in studies.items():
    row = {"Model":model_name, "Best_CV_AUC":study.best_value,
           "Best_CV_AUC_STD":study.best_trial.user_attrs.get("cv_auc_std")}
    row.update(study.best_params)
    best_rows.append(row)

    for p, v in study.best_params.items():
        best_long.append({
            "Model":model_name,
            "Best_CV_AUC":study.best_value,
            "Best_CV_AUC_STD":study.best_trial.user_attrs.get("cv_auc_std"),
            "Hyperparameter":p,
            "Selected_Value":v
        })

best_hyperparameters_wide_df = pd.DataFrame(best_rows)
best_hyperparameters_long_df = pd.DataFrame(best_long)

display(best_hyperparameters_wide_df)
display(best_hyperparameters_long_df)

,Model,Best_CV_AUC,Best_CV_AUC_STD,n_estimators,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda,min_samples_split,min_samples_leaf,max_features,bootstrap,class_weight
0,XGBoost,0.849530,0.014084,950,3,0.03919,4.982566,0.747168,0.657021,4.274154,9.840393e-07,0.177096,NaN,NaN,NaN,NaN,NaN
1,RandomForest,0.846856,0.014239,300,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.0,15.0,0.4,True,NaN
2,GBM,0.850365,0.013300,650,1,0.05147,NaN,0.793177,NaN,NaN,NaN,NaN,4.0,5.0,sqrt,NaN,NaN


,Model,Best_CV_AUC,Best_CV_AUC_STD,Hyperparameter,Selected_Value
0,XGBoost,0.849530,0.014084,n_estimators,950
1,XGBoost,0.849530,0.014084,max_depth,3
2,XGBoost,0.849530,0.014084,learning_rate,0.03919
3,XGBoost,0.849530,0.014084,min_child_weight,4.982566
4,XGBoost,0.849530,0.014084,subsample,0.747168
5,XGBoost,0.849530,0.014084,colsample_bytree,0.657021
6,XGBoost,0.849530,0.014084,gamma,4.274154
7,XGBoost,0.849530,0.014084,reg_alpha,0.000001
8,XGBoost,0.849530,0.014084,reg_lambda,0.177096
9,RandomForest,0.846856,0.014239,n_estimators,300


## 9. Export all Optuna trials for all four models

The complete Optuna optimization history is exported for:

- XGBoost
- Random Forest
- GBM
- CatBoost

This includes every trial, its objective value, trial state, optimized parameters, and stored CV AUC standard deviation where available.

In [12]:
studies = {
    "XGBoost": study_xgb,
    "RandomForest": study_rf,
    "GBM": study_gbm,
    "CatBoost": study_cat
}

all_trials_parts = []

for model_name, study in studies.items():
    trials_df_model = study.trials_dataframe(
        attrs=(
            "number",
            "value",
            "params",
            "user_attrs",
            "state"
        )
    ).copy()

    trials_df_model.insert(
        0,
        "Model",
        model_name
    )

    all_trials_parts.append(
        trials_df_model
    )

all_trials_df = pd.concat(
    all_trials_parts,
    ignore_index=True,
    sort=False
)

print(
    all_trials_df.groupby("Model")
    .size()
    .rename("Number_of_Trials")
)

display(all_trials_df.head(20))

Model
CatBoost        60
GBM             60
RandomForest    60
XGBoost         60
Name: Number_of_Trials, dtype: int64


,Model,number,value,params_colsample_bytree,params_gamma,params_learning_rate,params_max_depth,params_min_child_weight,params_n_estimators,params_reg_alpha,...,params_class_weight,params_max_features,params_min_samples_leaf,params_min_samples_split,params_bagging_temperature,params_border_count,params_depth,params_iterations,params_l2_leaf_reg,params_random_strength
0,XGBoost,0,0.843329,0.772736,2.383367,0.050378,10.0,5.033570,1000.0,2.353940e-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,XGBoost,1,0.846351,0.779305,1.425956,0.006298,6.0,5.488754,450.0,2.706690e-07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,XGBoost,2,0.846780,0.698682,3.589950,0.148075,7.0,12.539935,200.0,2.466852e-05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,XGBoost,3,0.843927,0.886407,0.420026,0.008500,3.0,13.398699,200.0,4.684179e-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,XGBoost,4,0.848750,0.872568,3.970089,0.069433,3.0,3.863793,750.0,2.713852e-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,XGBoost,5,0.846681,0.862100,2.290954,0.175197,2.0,3.393049,950.0,2.221779e-02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,XGBoost,6,0.847311,0.987737,3.125319,0.012978,7.0,8.826474,1150.0,1.390808e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,XGBoost,7,0.846670,0.669871,2.654432,0.008294,9.0,10.368221,950.0,1.415155e-04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,XGBoost,8,0.844786,0.858661,3.296088,0.068298,5.0,2.026639,1100.0,6.201891e-06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,XGBoost,9,0.847070,0.612106,4.134566,0.006006,9.0,3.116479,400.0,2.917635e-05,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
def fit_catboost_preprocessor(X):
    """Fit the simple frozen preprocessing needed by CatBoost on training data only."""
    numeric_cols = [
        c for c in X.columns
        if pd.api.types.is_numeric_dtype(X[c])
    ]
    categorical_cols = [
        c for c in X.columns
        if c not in numeric_cols
    ]
    numeric_medians = X[numeric_cols].median()
    return numeric_cols, categorical_cols, numeric_medians


def transform_catboost_frozen(
    X,
    numeric_cols,
    categorical_cols,
    numeric_medians
):
    X_cb = X.copy()

    X_cb[numeric_cols] = (
        X_cb[numeric_cols]
        .fillna(numeric_medians)
        .astype(float)
    )

    for c in categorical_cols:
        X_cb[c] = (
            X_cb[c]
            .fillna("__MISSING__")
            .astype(str)
        )

    return X_cb

## 10. Fit the four frozen models using Optuna-selected hyperparameters

Each final model is retrained on the full 70% training split using the best hyperparameters selected by its corresponding Optuna study.

- XGBoost uses `study_xgb.best_params`
- Random Forest uses `study_rf.best_params`
- GBM uses `study_gbm.best_params`
- CatBoost uses `study_cat.best_params`

The three sklearn/XGBoost models use the frozen OHE pipeline. CatBoost is fitted separately on the original categorical representation using training-only numeric medians and native categorical handling.

In [14]:
# Best Optuna hyperparameters
best_params = {
    "XGBoost": study_xgb.best_params.copy(),
    "RandomForest": study_rf.best_params.copy(),
    "GBM": study_gbm.best_params.copy(),
    "CatBoost": study_cat.best_params.copy()
}

# ------------------------------------------------------------------
# 1) OHE-based frozen models
# ------------------------------------------------------------------
xgb_estimator = XGBClassifier(
    **best_params["XGBoost"],
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=N_JOBS_MODEL
)

rf_estimator = RandomForestClassifier(
    **best_params["RandomForest"],
    random_state=SEED,
    n_jobs=N_JOBS_MODEL
)

gbm_estimator = GradientBoostingClassifier(
    **best_params["GBM"],
    random_state=SEED
)

models = {
    "XGBoost": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", xgb_estimator)
    ]),
    "RandomForest": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", rf_estimator)
    ]),
    "GBM": Pipeline([
        ("preprocessor", make_preprocessor()),
        ("model", gbm_estimator)
    ])
}

for name in ["XGBoost", "RandomForest", "GBM"]:
    print("Fitting:", name)
    models[name].fit(X_train, y_train)

# ------------------------------------------------------------------
# 2) CatBoost frozen model using native categoricals
# ------------------------------------------------------------------
(
    catboost_num_cols,
    catboost_cat_cols,
    catboost_numeric_medians
) = fit_catboost_preprocessor(X_train)

X_train_cb = transform_catboost_frozen(
    X_train,
    catboost_num_cols,
    catboost_cat_cols,
    catboost_numeric_medians
)

cat_features_idx = [
    X_train_cb.columns.get_loc(c)
    for c in catboost_cat_cols
]

catboost_estimator = CatBoostClassifier(
    **best_params["CatBoost"],
    loss_function="Logloss",
    eval_metric="AUC",
    verbose=0,
    random_seed=SEED,
    allow_writing_files=False,
    thread_count=N_JOBS_MODEL
)

print("Fitting: CatBoost")
catboost_estimator.fit(
    X_train_cb,
    y_train,
    cat_features=cat_features_idx
)

models["CatBoost"] = catboost_estimator

print("Frozen models:", list(models.keys()))

Fitting: XGBoost
Fitting: RandomForest
Fitting: GBM
Fitting: CatBoost
Frozen models: ['XGBoost', 'RandomForest', 'GBM', 'CatBoost']


## 11. Complete frozen-estimator parameter export

This is separate from the Optuna table.

- `Best Hyperparameters` = parameters that Optuna optimized.
- `Frozen Estimator Parameters` = complete final estimator configuration used in the experiment.

In [15]:
frozen_rows = []

for model_name, model in models.items():

    if model_name == "CatBoost":
        estimator_params = model.get_params()
    else:
        estimator_params = model.named_steps["model"].get_params(deep=False)

    for p, v in estimator_params.items():
        frozen_rows.append({
            "Model": model_name,
            "Parameter": p,
            "Value": str(v)
        })

frozen_estimator_parameters_df = pd.DataFrame(frozen_rows)
display(frozen_estimator_parameters_df)

,Model,Parameter,Value
0,XGBoost,objective,binary:logistic
1,XGBoost,base_score,None
2,XGBoost,booster,None
3,XGBoost,callbacks,None
4,XGBoost,colsample_bylevel,None
...,...,...,...
86,CatBoost,verbose,0
87,CatBoost,random_strength,3.4858913962894373
88,CatBoost,eval_metric,AUC
89,CatBoost,bagging_temperature,4.2688675051144465


### Threshold-selection helper

The function below must be defined before the validation-based threshold freezing step. It uses only the validation split and supports `f1` and `youden`.

In [18]:
def choose_threshold(y_true, probs, method="f1"):
    """
    Select the classification threshold using validation data only.
    Returns (best_threshold, best_score).
    """
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)

    method = method.lower()

    if method == "f1":
        candidate_thresholds = np.arange(0.05, 0.951, 0.005)

        scores = []
        for threshold in candidate_thresholds:
            pred = (probs >= threshold).astype(int)
            score = f1_score(
                y_true,
                pred,
                zero_division=0
            )
            scores.append(score)

        best_idx = int(np.argmax(scores))

        return (
            float(candidate_thresholds[best_idx]),
            float(scores[best_idx])
        )

    if method == "youden":
        fpr, tpr, thresholds = roc_curve(
            y_true,
            probs
        )

        youden_j = tpr - fpr
        best_idx = int(np.argmax(youden_j))

        return (
            float(thresholds[best_idx]),
            float(youden_j[best_idx])
        )

    raise ValueError(
        "THRESHOLD_METHOD must be 'f1' or 'youden'."
    )

## 12. Select and freeze one threshold per model using validation data only

In [19]:
def prepare_catboost_frozen(X):
    X_cb = X.copy()

    X_cb[catboost_num_cols] = (
        X_cb[catboost_num_cols]
        .fillna(catboost_numeric_medians)
        .astype(float)
    )

    for c in catboost_cat_cols:
        X_cb[c] = (
            X_cb[c]
            .fillna("__MISSING__")
            .astype(str)
        )

    return X_cb

In [20]:
thresholds = {}
threshold_rows = []

for model_name, model in models.items():

    if model_name == "CatBoost":
        X_valid_model = transform_catboost_frozen(
            X_valid,
            catboost_num_cols,
            catboost_cat_cols,
            catboost_numeric_medians
        )
    else:
        X_valid_model = X_valid

    probs = model.predict_proba(X_valid_model)[:, 1]

    t, score = choose_threshold(
        y_valid,
        probs,
        THRESHOLD_METHOD
    )

    thresholds[model_name] = t

    threshold_rows.append({
        "Model": model_name,
        "Threshold_Method": THRESHOLD_METHOD,
        "Frozen_Threshold": t,
        "Validation_Score": score
    })

frozen_thresholds_df = pd.DataFrame(threshold_rows)
display(frozen_thresholds_df)

,Model,Threshold_Method,Frozen_Threshold,Validation_Score
0,XGBoost,f1,0.380,0.656616
1,RandomForest,f1,0.380,0.642741
2,GBM,f1,0.365,0.661264
3,CatBoost,f1,0.380,0.656716


## 13. Baseline performance on untouched test data

In [21]:
def metrics(y_true, probs, threshold):
    pred = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1]
    ).ravel()

    return {
        "AUC": roc_auc_score(y_true, probs),
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "Threshold": threshold,
        "Actual_Churners": int((np.asarray(y_true) == 1).sum()),
        "Actual_NonChurners": int((np.asarray(y_true) == 0).sum()),
        "Predicted_Churners": int(pred.sum()),
        "Predicted_NonChurners": int((pred == 0).sum()),
        "True_Positives": int(tp),
        "False_Positives": int(fp),
        "False_Negatives": int(fn),
        "True_Negatives": int(tn),
        "Actual_Churners_Predicted": int(tp)
    }


baseline_probs = {}
baseline_pred = {}
baseline_rows = []

for model_name, model in models.items():

    if model_name == "CatBoost":
        X_test_model = transform_catboost_frozen(
            X_test,
            catboost_num_cols,
            catboost_cat_cols,
            catboost_numeric_medians
        )
    else:
        X_test_model = X_test

    probs = model.predict_proba(X_test_model)[:, 1]
    pred = probs >= thresholds[model_name]

    baseline_probs[model_name] = probs
    baseline_pred[model_name] = pred

    baseline_rows.append({
        "Model": model_name,
        **metrics(
            y_test,
            probs,
            thresholds[model_name]
        )
    })

baseline_performance_df = pd.DataFrame(baseline_rows)

display(
    baseline_performance_df[
        [
            "Model",
            "AUC",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "Threshold",
            "Actual_Churners",
            "Predicted_Churners",
            "True_Positives",
            "False_Positives",
            "False_Negatives",
            "True_Negatives"
        ]
    ]
)

,Model,AUC,Accuracy,Precision,Recall,F1,Threshold,Actual_Churners,Predicted_Churners,True_Positives,False_Positives,False_Negatives,True_Negatives
0,XGBoost,0.862109,0.793756,0.602606,0.658363,0.629252,0.380,281,307,185,122,96,654
1,RandomForest,0.857942,0.792810,0.600649,0.658363,0.628183,0.380,281,308,185,123,96,653
2,GBM,0.860818,0.794702,0.600629,0.679715,0.637730,0.365,281,318,191,127,90,649
3,CatBoost,0.861591,0.794702,0.606667,0.647687,0.626506,0.380,281,300,182,118,99,658


## 14. Parse Recommended Actions

In [22]:
CATEGORY_VALUES = {
    "PhoneService":["No","Yes"],
    "MultipleLines":["No","Yes","No phone service"],
    "InternetService":["DSL","Fiber optic","No"],
    "OnlineSecurity":["No","Yes","No internet service"],
    "OnlineBackup":["No","Yes","No internet service"],
    "DeviceProtection":["No","Yes","No internet service"],
    "TechSupport":["No","Yes","No internet service"],
    "StreamingTV":["No","Yes","No internet service"],
    "StreamingMovies":["No","Yes","No internet service"],
    "Contract":["Month-to-month","One year","Two year"],
    "PaymentMethod":["Electronic check","Mailed check","Bank transfer (automatic)","Credit card (automatic)"]
}

def norm(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def decode_source_state(source, target):
    if target not in CATEGORY_VALUES:
        return None
    if norm(source) == norm(target) and "Yes" in CATEGORY_VALUES[target]:
        return "Yes"
    for value in CATEGORY_VALUES[target]:
        if norm(value) in norm(source):
            return value
    return None

def normalize_recommended_value(target, value):
    mapping = {
        ("PaymentMethod","Credit card"):"Credit card (automatic)",
        ("PaymentMethod","Bank transfer"):"Bank transfer (automatic)"
    }
    return mapping.get((target,value), value)

def parse_action(row):
    text = str(row["Recommended_Action"]).strip()
    target = str(row["Action_Target"]).strip()
    candidate = "(candidate)" in text.lower()

    if target == "MonthlyCharges" and text.lower().startswith("reduce"):
        return {"kind":"reduce","target":target,"value":MONTHLY_CHARGES_REDUCTION_PCT,"candidate":candidate}

    m = re.search(r"=\s*(.+)$", text)
    if not m:
        raise ValueError(f"Cannot parse Recommended_Action for {row['Source']}: {text}")

    value = re.sub(r"\s*\(candidate\)\s*$","",m.group(1).strip(),flags=re.I)
    value = normalize_recommended_value(target,value)
    return {"kind":"set","target":target,"value":value,"candidate":candidate}

actions["_parsed"] = actions.apply(parse_action, axis=1)
actions["_source_state"] = actions.apply(
    lambda r: decode_source_state(r["Source"], r["Action_Target"]), axis=1
)

display(actions[["Source","Sign","Action_Target","Recommended_Action","Global Score","Priority","_source_state"]]
        .sort_values("Global Score", ascending=False))

,Source,Sign,Action_Target,Recommended_Action,Global Score,Priority,_source_state
0,InternetService_Fiber_optic,+,InternetService,InternetService = DSL (candidate),0.555556,HIGH,Fiber optic
1,Contract_Month-to-month,+,Contract,Contract = One year,0.500000,HIGH,Month-to-month
2,OnlineBackup_Yes,-,OnlineBackup,OnlineBackup = Yes,0.333333,HIGH,Yes
3,MonthlyCharges,+,MonthlyCharges,Reduce MonthlyCharges,0.333333,HIGH,NaN
4,PhoneService,-,PhoneService,PhoneService = Yes,0.250000,MEDIUM,Yes
5,MultipleLines_No_phone_service,+,PhoneService,PhoneService = Yes,0.250000,MEDIUM,No
6,InternetService_No,+,InternetService,InternetService = DSL (candidate),0.166667,MEDIUM,No
7,PaymentMethod_Mailed_check,-,PaymentMethod,PaymentMethod = Mailed check,0.166667,MEDIUM,Mailed check
8,PaymentMethod_Credit_card,-,PaymentMethod,PaymentMethod = Mailed check,0.083333,LOW,NaN
9,TechSupport_No_internet_service,+,InternetService,InternetService = DSL (candidate),0.083333,LOW,No


## 15. Apply intervention scenarios

In [23]:
SCENARIOS = {
    "HIGH":{"HIGH"},
    "HIGH+MEDIUM":{"HIGH","MEDIUM"},
    "HIGH+MEDIUM+LOW":{"HIGH","MEDIUM","LOW"}
}

def apply_scenario(df_original, baseline_churn_mask, allowed_priorities):
    modified = df_original.copy(deep=True)
    log_rows = []

    selected = actions[actions["Priority"].isin(allowed_priorities)].sort_values(
        ["Global Score","Global Rank"], ascending=[False,True]
    )

    already_changed = {
        target: np.zeros(len(modified), dtype=bool)
        for target in selected["Action_Target"].dropna().astype(str).unique()
        if target.strip()
    }

    for _, row in selected.iterrows():
        rec = row["_parsed"]

        if rec["candidate"] and not APPLY_CANDIDATE_ACTIONS:
            continue

        target = rec["target"]
        if target not in modified.columns:
            raise ValueError(f"Action_Target '{target}' is not a Telco column.")

        mask = baseline_churn_mask.copy()

        if row["Sign"] == "+":
            risky_state = row["_source_state"]
            if risky_state is not None:
                mask &= modified[target].astype(str).to_numpy() == str(risky_state)

        if target in already_changed:
            mask &= ~already_changed[target]

        if rec["kind"] == "set":
            mask &= modified[target].astype(str).to_numpy() != str(rec["value"])

            internet_dependent = {
                "OnlineSecurity","OnlineBackup","DeviceProtection",
                "TechSupport","StreamingTV","StreamingMovies"
            }

            if target in internet_dependent and rec["value"] == "Yes":
                mask &= modified["InternetService"].astype(str).to_numpy() != "No"

            if target == "MultipleLines" and rec["value"] == "Yes":
                mask &= modified["PhoneService"].astype(str).to_numpy() == "Yes"

            modified.loc[mask,target] = rec["value"]

        elif rec["kind"] == "reduce":
            modified.loc[mask,target] = (
                modified.loc[mask,target].astype(float) * (1.0 - rec["value"])
            )

        if target in already_changed:
            already_changed[target] |= mask

        log_rows.append({
            "Source":row["Source"], "Sign":row["Sign"], "Action_Target":target,
            "Recommended_Action":row["Recommended_Action"],
            "Global Score":row["Global Score"], "Global Rank":row["Global Rank"],
            "Priority":row["Priority"], "Changed_Customers":int(mask.sum())
        })

    return modified, pd.DataFrame(log_rows)

### Ground-truth check and counterfactual limitation

The observed test-set `Churn` value is used for **baseline predictive validation** and to distinguish baseline-predicted churners who are **actual churners (true positives)** from baseline false positives.

After predictors are changed to simulate an intervention, the true post-intervention outcome is not observed in this dataset. Therefore, the original `Churn` value is **not** treated as post-intervention ground truth. `Predicted Yes → No` and **Model-predicted rescue rate** describe changes in the frozen model's output, not customers proven to have been retained in reality.

A separate result is reported for actual churners correctly identified at baseline, providing a stronger link to observed reality without claiming that their counterfactual outcomes are known.


## 16. Run 4 frozen models × 3 intervention scenarios

The intervention experiment is run independently for each frozen predictive model:

- XGBoost
- Random Forest
- GBM
- CatBoost

and for each intervention scenario:

- HIGH
- HIGH + MEDIUM
- HIGH + MEDIUM + LOW

Therefore, the experiment contains:

$$
4 \text{ models} \times 3 \text{ scenarios} = 12 \text{ intervention experiments}
$$

Each model uses its own frozen validation-selected classification threshold and its own baseline-predicted churner population.

The **primary impact analysis** is restricted to actual churners correctly predicted at baseline (True Positives). False Positives are retained as a separate diagnostic subgroup and are not counted as rescued customers.

In [24]:
scenario_rows, customer_rows, application_logs = [], [], []
actual_churn = y_test.to_numpy().astype(bool)

for model_name, model in models.items():
    before_probs = baseline_probs[model_name]
    before_pred = baseline_pred[model_name]
    threshold = thresholds[model_name]

    # Baseline groups
    tp_mask = before_pred & actual_churn
    fp_mask = before_pred & (~actual_churn)

    for scenario_name, priority_set in SCENARIOS.items():
        modified_test_df, app_log = apply_scenario(
            test_df, before_pred.copy(), priority_set
        )
        X_modified = modified_test_df.drop(columns=[TARGET, ID_COL])
        if model_name == "CatBoost":
            X_modified_model = transform_catboost_frozen(
                X_modified,
                catboost_num_cols,
                catboost_cat_cols,
                catboost_numeric_medians
            )
        else:
            X_modified_model = X_modified

        after_probs = model.predict_proba(
            X_modified_model
        )[:, 1]
        after_pred = after_probs >= threshold
        delta = before_probs - after_probs

        # PRIMARY: real churners correctly predicted at baseline
        tp_yes_to_no = tp_mask & (~after_pred)
        idx_tp = np.where(tp_mask)[0]

        # SECONDARY diagnostic: baseline false positives
        fp_yes_to_no = fp_mask & (~after_pred)
        idx_fp = np.where(fp_mask)[0]

        # Secondary overall predicted-churner view
        all_yes_to_no = before_pred & (~after_pred)
        idx_all = np.where(before_pred)[0]

        scenario_rows.append({
            "Model": model_name,
            "Scenario": scenario_name,
            "Frozen_Threshold": threshold,

            "Baseline_Predicted_Churners": int(before_pred.sum()),
            "Actual_Churners_Correctly_Predicted_TP": int(tp_mask.sum()),
            "Baseline_False_Positives": int(fp_mask.sum()),

            # PRIMARY impact metrics
            "Predicted_Yes_to_No_Among_Actual_Churners": int(tp_yes_to_no.sum()),
            "Model_Predicted_Rescue_Rate_Actual_Churners": (
                float(tp_yes_to_no.sum() / tp_mask.sum()) if tp_mask.sum() else np.nan
            ),
            "Mean_Delta_P_Churn_Actual_Churners": (
                float(delta[idx_tp].mean()) if len(idx_tp) else np.nan
            ),
            "Median_Delta_P_Churn_Actual_Churners": (
                float(np.median(delta[idx_tp])) if len(idx_tp) else np.nan
            ),
            "Mean_P_Before_Actual_Churners": (
                float(before_probs[idx_tp].mean()) if len(idx_tp) else np.nan
            ),
            "Mean_P_After_Actual_Churners": (
                float(after_probs[idx_tp].mean()) if len(idx_tp) else np.nan
            ),
            "Worsened_Actual_Churners": int(
                (tp_mask & (after_probs > before_probs + 1e-12)).sum()
            ),

            # FP diagnostic only -- NOT rescues
            "FP_Predicted_Yes_to_No": int(fp_yes_to_no.sum()),
            "FP_Transition_Rate": (
                float(fp_yes_to_no.sum() / fp_mask.sum()) if fp_mask.sum() else np.nan
            ),
            "Mean_Delta_P_Churn_False_Positives": (
                float(delta[idx_fp].mean()) if len(idx_fp) else np.nan
            ),

            # Secondary all-predicted-churner metrics
            "All_Predicted_Yes_to_No": int(all_yes_to_no.sum()),
            "All_Predicted_Transition_Rate": (
                float(all_yes_to_no.sum() / before_pred.sum()) if before_pred.sum() else np.nan
            ),
            "Mean_Delta_P_Churn_All_Predicted_Churners": (
                float(delta[idx_all].mean()) if len(idx_all) else np.nan
            )
        })

        customer_rows.append(pd.DataFrame({
            ID_COL: test_df[ID_COL].to_numpy()[idx_all],
            "Model": model_name,
            "Scenario": scenario_name,
            "Actual_Churn": test_df[TARGET].to_numpy()[idx_all],
            "Baseline_Group": np.where(
                actual_churn[idx_all],
                "True Positive - actual churner",
                "False Positive - actual non-churner"
            ),
            "P_Churn_Before": before_probs[idx_all],
            "P_Churn_After": after_probs[idx_all],
            "Delta_P_Churn": delta[idx_all],
            "Predicted_After": np.where(after_pred[idx_all], "Yes", "No"),
            "TP_Model_Predicted_Rescue": np.where(
                actual_churn[idx_all] & (~after_pred[idx_all]), "Yes", "No"
            ),
            "FP_Yes_to_No_Diagnostic": np.where(
                (~actual_churn[idx_all]) & (~after_pred[idx_all]), "Yes", "No"
            )
        }))

        if not app_log.empty:
            app_log.insert(0, "Model", model_name)
            app_log.insert(1, "Scenario", scenario_name)
            application_logs.append(app_log)

scenario_summary_df = pd.DataFrame(scenario_rows)
customer_level_df = pd.concat(customer_rows, ignore_index=True)
application_log_df = (
    pd.concat(application_logs, ignore_index=True)
    if application_logs else pd.DataFrame()
)

display(scenario_summary_df[[
    "Model", "Scenario",
    "Actual_Churners_Correctly_Predicted_TP",
    "Predicted_Yes_to_No_Among_Actual_Churners",
    "Model_Predicted_Rescue_Rate_Actual_Churners",
    "Mean_Delta_P_Churn_Actual_Churners",
    "Baseline_False_Positives",
    "FP_Predicted_Yes_to_No",
    "FP_Transition_Rate"
]])

,Model,Scenario,Actual_Churners_Correctly_Predicted_TP,Predicted_Yes_to_No_Among_Actual_Churners,Model_Predicted_Rescue_Rate_Actual_Churners,Mean_Delta_P_Churn_Actual_Churners,Baseline_False_Positives,FP_Predicted_Yes_to_No,FP_Transition_Rate
0,XGBoost,HIGH,185,128,0.691892,0.302904,122,113,0.926230
1,XGBoost,HIGH+MEDIUM,185,147,0.794595,0.346630,122,116,0.950820
2,XGBoost,HIGH+MEDIUM+LOW,185,164,0.886486,0.377145,122,120,0.983607
3,RandomForest,HIGH,185,126,0.681081,0.280429,123,112,0.910569
4,RandomForest,HIGH+MEDIUM,185,133,0.718919,0.302449,123,111,0.902439
5,RandomForest,HIGH+MEDIUM+LOW,185,156,0.843243,0.331799,123,119,0.967480
6,GBM,HIGH,191,98,0.513089,0.226664,127,92,0.724409
7,GBM,HIGH+MEDIUM,191,142,0.743455,0.303953,127,113,0.889764
8,GBM,HIGH+MEDIUM+LOW,191,151,0.790576,0.334421,127,120,0.944882
9,CatBoost,HIGH,182,100,0.549451,0.248453,118,90,0.762712


## 17. Results table

In [25]:
paper_results_table = scenario_summary_df[[
    "Model",
    "Scenario",
    "Actual_Churners_Correctly_Predicted_TP",
    "Predicted_Yes_to_No_Among_Actual_Churners",
    "Model_Predicted_Rescue_Rate_Actual_Churners",
    "Mean_Delta_P_Churn_Actual_Churners",
    "Median_Delta_P_Churn_Actual_Churners",
    "Worsened_Actual_Churners"
]].rename(columns={
    "Model": "Frozen model",
    "Actual_Churners_Correctly_Predicted_TP": "Actual churners correctly predicted (TP)",
    "Predicted_Yes_to_No_Among_Actual_Churners": "Predicted Yes → No among actual churners",
    "Model_Predicted_Rescue_Rate_Actual_Churners": "Model-predicted rescue rate",
    "Mean_Delta_P_Churn_Actual_Churners": "Mean Δ churn probability",
    "Median_Delta_P_Churn_Actual_Churners": "Median Δ churn probability",
    "Worsened_Actual_Churners": "Actual churners with increased predicted risk"
}).copy()

model_order = ["XGBoost", "RandomForest", "GBM", "CatBoost"]
scenario_order = ["HIGH", "HIGH+MEDIUM", "HIGH+MEDIUM+LOW"]

paper_results_table["Frozen model"] = pd.Categorical(
    paper_results_table["Frozen model"], categories=model_order, ordered=True
)
paper_results_table["Scenario"] = pd.Categorical(
    paper_results_table["Scenario"], categories=scenario_order, ordered=True
)
paper_results_table = paper_results_table.sort_values(
    ["Frozen model", "Scenario"]
).reset_index(drop=True)

paper_results_display = paper_results_table.copy()
paper_results_display["Model-predicted rescue rate"] = (
    paper_results_display["Model-predicted rescue rate"] * 100
).map(lambda x: f"{x:.2f}%")

for col in ["Mean Δ churn probability", "Median Δ churn probability"]:
    paper_results_display[col] = paper_results_display[col].map(lambda x: f"{x:.4f}")

display(paper_results_display)

,Frozen model,Scenario,Actual churners correctly predicted (TP),Predicted Yes → No among actual churners,Model-predicted rescue rate,Mean Δ churn probability,Median Δ churn probability,Actual churners with increased predicted risk
0,XGBoost,HIGH,185,128,69.19%,0.3029,0.3134,0
1,XGBoost,HIGH+MEDIUM,185,147,79.46%,0.3466,0.3483,0
2,XGBoost,HIGH+MEDIUM+LOW,185,164,88.65%,0.3771,0.3802,0
3,RandomForest,HIGH,185,126,68.11%,0.2804,0.2749,0
4,RandomForest,HIGH+MEDIUM,185,133,71.89%,0.3024,0.2983,0
5,RandomForest,HIGH+MEDIUM+LOW,185,156,84.32%,0.3318,0.3274,0
6,GBM,HIGH,191,98,51.31%,0.2267,0.2359,0
7,GBM,HIGH+MEDIUM,191,142,74.35%,0.3040,0.3197,0
8,GBM,HIGH+MEDIUM+LOW,191,151,79.06%,0.3344,0.3442,0
9,CatBoost,HIGH,182,100,54.95%,0.2485,0.2461,0


## FP Diagnostic Table

False Positives are reported separately. A baseline FP that changes from predicted `Yes` to predicted `No` is **not counted as a rescue**, because its observed outcome is already `Churn = No`.

In [26]:
fp_diagnostic_table = scenario_summary_df[[
    "Model",
    "Scenario",
    "Baseline_False_Positives",
    "FP_Predicted_Yes_to_No",
    "FP_Transition_Rate",
    "Mean_Delta_P_Churn_False_Positives"
]].copy()

display(fp_diagnostic_table)

,Model,Scenario,Baseline_False_Positives,FP_Predicted_Yes_to_No,FP_Transition_Rate,Mean_Delta_P_Churn_False_Positives
0,XGBoost,HIGH,122,113,0.926230,0.283672
1,XGBoost,HIGH+MEDIUM,122,116,0.950820,0.324167
2,XGBoost,HIGH+MEDIUM+LOW,122,120,0.983607,0.346259
3,RandomForest,HIGH,123,112,0.910569,0.265716
4,RandomForest,HIGH+MEDIUM,123,111,0.902439,0.283204
5,RandomForest,HIGH+MEDIUM+LOW,123,119,0.967480,0.305226
6,GBM,HIGH,127,92,0.724409,0.222802
7,GBM,HIGH+MEDIUM,127,113,0.889764,0.294356
8,GBM,HIGH+MEDIUM+LOW,127,120,0.944882,0.319462
9,CatBoost,HIGH,118,90,0.762712,0.241392


## 18. Export Excel results

In [27]:
OUTPUT_FILE = "Telco_Optuna_Intervention_Results.xlsx"

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    search_spaces_df.to_excel(writer, sheet_name="Search Spaces", index=False)
    all_trials_df.to_excel(writer, sheet_name="All Optuna Trials", index=False)
    best_hyperparameters_wide_df.to_excel(writer, sheet_name="Best Params Wide", index=False)
    best_hyperparameters_long_df.to_excel(writer, sheet_name="Best Hyperparameters", index=False)
    frozen_estimator_parameters_df.to_excel(writer, sheet_name="Frozen Estimator Params", index=False)
    frozen_thresholds_df.to_excel(writer, sheet_name="Frozen Thresholds", index=False)
    baseline_performance_df.to_excel(writer, sheet_name="Baseline Performance", index=False)
    scenario_summary_df.to_excel(writer, sheet_name="Scenario Summary", index=False)
    fp_diagnostic_table.to_excel(writer, sheet_name="FP Diagnostic", index=False)
    paper_results_table.to_excel(writer, sheet_name="Main Results Table", index=False)
    customer_level_df.to_excel(writer, sheet_name="Customer Level", index=False)
    application_log_df.to_excel(writer, sheet_name="Action Application Log", index=False)
    actions.drop(columns=["_parsed"]).to_excel(writer, sheet_name="Prioritized Actions", index=False)

print("Saved:", OUTPUT_FILE)

Saved: Telco_Optuna_Intervention_Results.xlsx


## 19. Save frozen pipelines and complete experiment metadata

### Model persistence

All four frozen predictive models are saved consistently using `joblib`:

- `XGBoost.joblib`
- `RandomForest.joblib`
- `GBM.joblib`
- `CatBoost.joblib`

For CatBoost, the training-derived numeric medians and the lists of numeric and native categorical columns remain stored in `experiment_metadata.json`. These values are required to reproduce the same frozen CatBoost preprocessing before prediction.

In [29]:
MODEL_DIR = Path("frozen_models")

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Save all four frozen predictive models consistently with joblib.
for model_name, model in models.items():
    joblib.dump(
        model,
        MODEL_DIR / f"{model_name}.joblib"
    )

# CatBoost uses separate frozen preprocessing metadata because its native
# categorical representation is intentionally not wrapped in the OHE pipeline.
metadata = {
    "seed": SEED,
    "threshold_method": THRESHOLD_METHOD,
    "thresholds": {
        k: float(v)
        for k, v in thresholds.items()
    },
    "models": list(models.keys()),
    "model_files": {
        model_name: f"{model_name}.joblib"
        for model_name in models.keys()
    },
    "catboost_native_categoricals": catboost_cat_cols,
    "catboost_numeric_columns": catboost_num_cols,
    "catboost_numeric_medians": {
        k: float(v)
        for k, v in catboost_numeric_medians.items()
    }
}

with open(
    MODEL_DIR / "experiment_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved frozen models:")
for model_name in models.keys():
    print(" -", MODEL_DIR / f"{model_name}.joblib")

print("Saved metadata:")
print(" -", MODEL_DIR / "experiment_metadata.json")

Saved frozen models:
 - frozen_models\XGBoost.joblib
 - frozen_models\RandomForest.joblib
 - frozen_models\GBM.joblib
 - frozen_models\CatBoost.joblib
Saved metadata:
 - frozen_models\experiment_metadata.json


## Main outcome definitions and interpretation

The primary population is:

$$
TP=\{i:Y_i=1 \land \hat{Y}^{baseline}_i=1\}.
$$

For each correctly identified actual churner:

$$
\Delta p_i=p_i^{baseline}-p_i^{intervention}.
$$

The main impact metric is:

$$
\boxed{
Model\text{-}predicted\ rescue\ rate
=
\frac{\#(Y=1 \land \hat{Y}^{baseline}=1 \land \hat{Y}^{intervention}=0)}
{\#(Y=1 \land \hat{Y}^{baseline}=1)}
}
$$

This is **not** an observed real-world retention rate, because the dataset does not contain the true post-intervention outcome for the same customer.

False Positives are explicitly excluded from the rescue metric. Their Yes → No transitions are reported only as a diagnostic analysis.